In [1]:
  from google.colab import drive
  # Tenta desmontar o Drive para garantir uma nova conexão limpa
  try:
      drive.flush_and_unmount()
  except:
      pass
  # Monta o Drive novamente. Você deve ver um link e pedir a permissão.
  drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [2]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem, Caminhos e Limpeza (ADAPTADO PARA TXT E CSV)
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Assumindo que o drive.mount foi executado com sucesso em uma célula anterior
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada (ADAPTADO PARA TXT/CSV) ---
NOME_ARQUIVO_ENTRADA = input("Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): ").strip()

# Se o usuário não forneceu nenhuma extensão, assume .txt
if '.' not in NOME_ARQUIVO_ENTRADA:
    NOME_ARQUIVO_ENTRADA += '.txt'

ARQUIVO_ENTRADA = NOME_ARQUIVO_ENTRADA
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA)

# Define o nome "curto" para logs e mensagens
NOME_LOG = ARQUIVO_ENTRADA.rsplit('.', 1)[0].upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# --- LIMPEZA DE RESULTADOS ANTERIORES ---
if os.path.exists(CAMINHO_SAVE_ESTRUTURADO):
    try:
        os.remove(CAMINHO_SAVE_ESTRUTURADO)
        print(f"✅ Arquivo de resultados anteriores removido: {os.path.basename(CAMINHO_SAVE_ESTRUTURADO)}")
    except Exception as e:
        print(f"AVISO: Não foi possível remover o arquivo de resultados anteriores. {e}")
print("-" * 50)


# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO (COM AJUSTE PARA CSV/TABULAR)
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    # Garante que a coluna 'Dispositivo' exista, crucial para o groupby posterior.
    if 'Dispositivo' not in df.columns:
        df['Dispositivo'] = nome_log

    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial.")
        return False

    df.set_index('Timestamp', inplace=True)

    # Forçar tipos numéricos
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando o padrão REGEX (para logs não estruturados)."""
    print(f"\nIniciando estruturação (REGEX) de: {os.path.basename(caminho_entrada)}")
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado: {caminho_entrada}")
        return False
    except Exception as e:
        print(f"ERRO de leitura REGEX: {e}")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"AVISO REGEX: Nenhuma linha de dados válida encontrada em {nome_log}.")
        return False

    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.rename(columns={'Temp': 'Temperatura', 'Umid': 'Umidade'}, inplace=True)

    return processar_dataframe(df, nome_log, caminho_saida)


def estruturar_log_csv(caminho_entrada, caminho_saida, nome_log):
    """
    Tenta estruturar focando no formato CSV com Cabeçalho e Vírgula (Formato DADOS_dia.CSV).
    Inclui fallback para o formato antigo (tabular/sem cabeçalho).
    """
    print(f"\nTentando estruturação (CSV/Tabular) de: {os.path.basename(caminho_entrada)}")

    df = None
    # Priorizamos a vírgula para CSVs com cabeçalho.
    separadores = [(',', 'CSV (,)')]
    # Adicionamos o tabular (\t) como fallback para .txt ou CSVs antigos sem cabeçalho
    separadores.append(('\t', 'Tabular (\\t)'))

    for sep_char, sep_nome in separadores:
        try:
            # === TENTA LEITURA COM CABEÇALHO (Para DADOS_dia.CSV) ===
            df_teste = pd.read_csv(caminho_entrada, sep=sep_char, engine='python', on_bad_lines='skip')

            # Verificação básica de colunas
            if df_teste.shape[1] >= 5:
                # Tenta mapear as colunas do seu CSV
                colunas_presentes = set(df_teste.columns)
                if 'Temperatura(C)' in colunas_presentes and 'Umidade(%)' in colunas_presentes:
                    df = df_teste.copy()
                    print(f"INFO: Usando formato delimitado ({sep_nome}) com cabeçalho para {nome_log}")

                    # 1. Renomear/Mapear as colunas
                    df.rename(columns={
                        'Temperatura(C)': 'Temperatura',
                        'Umidade(%)': 'Umidade'
                    }, inplace=True)

                    # 2. Criar a coluna de Timestamp combinada
                    df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
                    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

                    # 3. Definir a coluna Dispositivo
                    df['Dispositivo'] = nome_log

                    # 4. Selecionar apenas as colunas de interesse
                    df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()
                    break # Sucesso na leitura com cabeçalho

            # === TENTA LEITURA SEM CABEÇALHO (Para S1.txt, etc.) ===
            col_names = ['Col_ID_Temporario', 'Data', 'Hora', 'Temperatura', 'Umidade']
            df_teste = pd.read_csv(caminho_entrada, sep=sep_char, header=None,
                                   names=col_names,
                                   engine='python', on_bad_lines='skip')

            if df_teste.shape[1] >= 5 and df_teste.shape[0] > 0:
                df = df_teste.iloc[:, :5].copy()
                print(f"INFO: Usando formato delimitado ({sep_nome}) sem cabeçalho para {nome_log}")

                # Lógica para forçar o ID do dispositivo
                if df['Col_ID_Temporario'].astype(str).str.isnumeric().all():
                    df['Dispositivo'] = nome_log
                else:
                    df.rename(columns={'Col_ID_Temporario': 'Dispositivo'}, inplace=True)

                # Converte Data e Hora para Timestamp
                df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
                df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
                break

        except Exception:
            continue

    if df is None or df.empty:
        print(f"AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com nenhum delimitador. Verifique a estrutura do arquivo.")
        return False

    # Processamento final
    return processar_dataframe(df, nome_log, caminho_saida)


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG)


# ====================================================================
# 3. Carregamento Final
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    exit()
except Exception:
    pass

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo de entrada.")
    exit()
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS
# ====================================================================

resultados_analise = []
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    media_temp = df_grupo['Temp'].mean()
    media_umid = df_grupo['Umid'].mean()

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Aritmetica': media_temp,
        'Umid_Média_Aritmetica': media_umid,
        'Total_Pontos': len(df_grupo)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados
# ====================================================================

# 1. Cálculo da Média Aritmética GERAL
media_temp_geral = df_dados['Temp'].mean()
media_umid_geral = df_dados['Umid'].mean()

# 2. Apresentação do Resultado FINAL (Geral)
print("\n" + "="*90)
print(f"|   ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: {ARQUIVO_ENTRADA}    |")
print("="*90)

print("\n--- 📊 Média ARITMÉTICA por Dispositivo (Individual) ---")
print(df_resultados[['Temp_Média_Aritmetica', 'Umid_Média_Aritmetica', 'Total_Pontos']].to_markdown(floatfmt=".2f"))


print("\n--- 🌟 Média ARITMÉTICA GERAL do Arquivo ---")
print(f"|   Temperatura Média GERAL: {media_temp_geral:.2f}°C")
print(f"|   Umidade Média GERAL:     {media_umid_geral:.2f}%")
print("------------------------------------------")

Montando Google Drive...
Drive já montado.
Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): DADOS_06_10 _1.CSV
Caminho do arquivo de entrada definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/DADOS_06_10 _1.CSV
Caminho de saída estruturado definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/dados_estruturados.csv
✅ Arquivo de resultados anteriores removido: dados_estruturados.csv
--------------------------------------------------

Iniciando estruturação (REGEX) de: DADOS_06_10 _1.CSV
ERRO: Arquivo não encontrado: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/DADOS_06_10 _1.CSV

Tentando estruturação (CSV/Tabular) de: DADOS_06_10 _1.CSV
AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com nenhum delimitador. Verifique a estrutura do arquivo.

ERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.

ERRO FATAL: O DataFrame DADOS_06_10 _1 está vazio. Verifique o formato do arquivo de e

KeyError: 'Dispositivo'

In [1]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem, Caminhos e Limpeza (ADAPTADO PARA TXT E CSV)
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # A montagem deve ser feita em uma célula anterior, mas podemos incluir a chamada
    # drive.mount('/content/drive')
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada (ADAPTADO PARA TXT/CSV) ---
NOME_ARQUIVO_ENTRADA = input("Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): ").strip()
DELIMITADOR_ENTRADA = input("Digite o delimitador do arquivo (Ex: , ou ; ou \\t para tabulação, ENTER para Vírgula): ").strip()
if not DELIMITADOR_ENTRADA:
    DELIMITADOR_ENTRADA = ',' # Padrão para CSV

# Se o usuário não forneceu nenhuma extensão, tenta adivinhar (mantido do original)
if '.' not in NOME_ARQUIVO_ENTRADA:
    NOME_ARQUIVO_ENTRADA += '.txt'

ARQUIVO_ENTRADA = NOME_ARQUIVO_ENTRADA
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA)

# Define o nome "curto" para logs e mensagens
NOME_LOG = ARQUIVO_ENTRADA.rsplit('.', 1)[0].upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Delimitador selecionado: {DELIMITADOR_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# --- LIMPEZA DE RESULTADOS ANTERIORES ---
if os.path.exists(CAMINHO_SAVE_ESTRUTURADO):
    try:
        os.remove(CAMINHO_SAVE_ESTRUTURADO)
        print(f"✅ Arquivo de resultados anteriores removido: {os.path.basename(CAMINHO_SAVE_ESTRUTURADO)}")
    except Exception as e:
        print(f"AVISO: Não foi possível remover o arquivo de resultados anteriores. {e}")
print("-" * 50)


# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO (COM AJUSTE PARA CSV/TABULAR)
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    # Garante que a coluna 'Dispositivo' exista, crucial para o groupby posterior.
    if 'Dispositivo' not in df.columns:
        df['Dispositivo'] = nome_log

    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial.")
        return False

    df.set_index('Timestamp', inplace=True)

    # Forçar tipos numéricos
    # A lógica original do CSV renomeia para 'Temperatura' e 'Umidade' antes de chegar aqui
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando o padrão REGEX (para logs não estruturados)."""
    print(f"\nIniciando estruturação (REGEX) de: {os.path.basename(caminho_entrada)}")
    # Padrão REGEX original mantido para logs não-tabulares
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado: {caminho_entrada}")
        return False
    except Exception as e:
        print(f"ERRO de leitura REGEX: {e}")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"AVISO REGEX: Nenhuma linha de dados válida encontrada em {nome_log}.")
        return False

    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.rename(columns={'Temp': 'Temperatura', 'Umid': 'Umidade'}, inplace=True)

    # A coluna 'Dispositivo' já vem do REGEX se o padrão for encontrado
    return processar_dataframe(df, nome_log, caminho_saida)


def estruturar_log_csv(caminho_entrada, caminho_saida, nome_log, delimitador):
    """
    Tenta estruturar dados tabulares (CSV/TXT) usando o delimitador especificado.
    Prioriza a leitura com cabeçalho, mas tem fallback para leitura sem cabeçalho.
    """
    print(f"\nTentando estruturação (CSV/Tabular com delimitador '{delimitador}') de: {os.path.basename(caminho_entrada)}")

    df = None
    df_raw = None

    try:
        # === 1. Tenta Leitura ASSUMINDO CABEÇALHO ===
        df_raw = pd.read_csv(caminho_entrada, sep=delimitador, engine='python', on_bad_lines='skip')
        print(f"INFO: Leitura com cabeçalho bem-sucedida. Colunas lidas: {list(df_raw.columns)}")

        # === 2. Tenta Mapear Colunas Comuns ===
        # Dicionário de renomeação para unificar nomes
        col_map = {
            'Temperatura(C)': 'Temperatura',
            'Umidade(%)': 'Umidade',
            'Temp': 'Temperatura',
            'Umid': 'Umidade',
            'temperature': 'Temperatura',
            'humidity': 'Umidade'
        }
        df = df_raw.rename(columns=col_map)

        # Tentativa de criar 'Timestamp' unificado, buscando por colunas 'Data' e 'Hora'
        if 'Data' in df.columns and 'Hora' in df.columns:
            df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
        elif 'Timestamp' not in df.columns and 'timestamp' in df.columns:
             df.rename(columns={'timestamp': 'Timestamp'}, inplace=True)

        # Se as colunas essenciais não existirem, passa para o fallback sem cabeçalho
        if not all(col in df.columns for col in ['Temperatura', 'Umidade', 'Timestamp']):
            df = None # Força a ir para o bloco de erro/fallback

    except Exception as e:
        print(f"AVISO: Leitura com cabeçalho e delimitador '{delimitador}' falhou. Tentando leitura sem cabeçalho.")
        # print(f"Detalhes do erro: {e}")
        pass # Ignora o erro e segue para o fallback sem cabeçalho

    if df is None or not all(col in df.columns for col in ['Temperatura', 'Umidade', 'Timestamp']):
        # === 3. Tenta Leitura SEM CABEÇALHO (Fallback) ===
        col_names = ['Col_0', 'Data', 'Hora', 'Temperatura', 'Umidade'] # Nomes temporários
        try:
            df = pd.read_csv(caminho_entrada, sep=delimitador, header=None,
                             names=col_names,
                             engine='python', on_bad_lines='skip')

            # Ajuste de colunas e criação do Timestamp para o formato 'S1.txt' (tabular/sem cabeçalho)
            if df.shape[1] >= 5 and df.shape[0] > 0:
                df = df.iloc[:, :5].copy()
                df.rename(columns={'Col_0': 'Dispositivo'}, inplace=True)

                # Assume que as colunas 1 e 2 são Data e Hora
                df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
                print(f"INFO: Usando formato delimitado ('{delimitador}') **sem cabeçalho** para {nome_log}")
            else:
                 raise ValueError("CSV lido sem cabeçalho não possui colunas suficientes.")

        except Exception:
            df = None # Confirma que a leitura falhou

    if df is None or df.empty:
        print(f"AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com delimitador '{delimitador}'. Verifique a estrutura.")
        return False

    # Converte para datetime e força o dispositivo se necessário
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    if 'Dispositivo' not in df.columns:
        df['Dispositivo'] = nome_log

    # Selecionar e processar
    df = df[['Dispositivo', 'Timestamp', 'Temperatura', 'Umidade']].copy()
    return processar_dataframe(df, nome_log, caminho_saida)


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

# 1. Tenta REGEX (para logs não estruturados)
if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    # 2. Tenta CSV/Tabular com o delimitador fornecido
    if not estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG, DELIMITADOR_ENTRADA):
        # 3. Tenta CSV/Tabular com o ponto e vírgula (fallback comum para CSVs brasileiros)
        if DELIMITADOR_ENTRADA != ';':
            print("Tentando fallback com delimitador ';'")
            estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG, ';')
        else:
             print("Nenhuma das tentativas de estruturação foi bem-sucedida.")


# ====================================================================
# 3. Carregamento Final (O restante do código não precisa de alterações)
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    exit()
except Exception:
    pass

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo de entrada.")
    exit()
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS
# ====================================================================

resultados_analise = []
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    media_temp = df_grupo['Temp'].mean()
    media_umid = df_grupo['Umid'].mean()

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Aritmetica': media_temp,
        'Umid_Média_Aritmetica': media_umid,
        'Total_Pontos': len(df_grupo)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados
# ====================================================================

# 1. Cálculo da Média Aritmética GERAL
media_temp_geral = df_dados['Temp'].mean()
media_umid_geral = df_dados['Umid'].mean()

# 2. Apresentação do Resultado FINAL (Geral)
print("\n" + "="*90)
print(f"|   ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: {ARQUIVO_ENTRADA}    |")
print("="*90)

print("\n--- 📊 Média ARITMÉTICA por Dispositivo (Individual) ---")
print(df_resultados[['Temp_Média_Aritmetica', 'Umid_Média_Aritmetica', 'Total_Pontos']].to_markdown(floatfmt=".2f"))


print("\n--- 🌟 Média ARITMÉTICA GERAL do Arquivo ---")
print(f"|   Temperatura Média GERAL: {media_temp_geral:.2f}°C")
print(f"|   Umidade Média GERAL:     {media_umid_geral:.2f}%")
print("------------------------------------------")

Montando Google Drive...
Drive já montado.
Digite o nome do arquivo TXT ou CSV para análise (Ex: PicoW.txt, S1.csv): DADOS_06_10 _1.CSV
Digite o delimitador do arquivo (Ex: , ou ; ou \t para tabulação, ENTER para Vírgula): DADOS_06_10 _1.CSV
Caminho do arquivo de entrada definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/DADOS_06_10 _1.CSV
Delimitador selecionado: DADOS_06_10 _1.CSV
Caminho de saída estruturado definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/dados_estruturados.csv
--------------------------------------------------

Iniciando estruturação (REGEX) de: DADOS_06_10 _1.CSV
ERRO: Arquivo não encontrado: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/DADOS_06_10 _1.CSV

Tentando estruturação (CSV/Tabular com delimitador 'DADOS_06_10 _1.CSV') de: DADOS_06_10 _1.CSV
AVISO: Leitura com cabeçalho e delimitador 'DADOS_06_10 _1.CSV' falhou. Tentando leitura sem cabeçalho.
AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente 

KeyError: 'Dispositivo'